# Длительный эксперимент A100
Читает промежуточные результаты и итоговую оценку. Для локального просмотра скопируйте папку запуска с сервера в eda/experiments. Development score используется для выбора; evaluation — только для отчёта.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import display
here=Path.cwd().resolve()
eda=here if here.name=='eda' else here/'eda'
run=eda/'experiments'/'overnight_20260915_13h'
display(json.loads((run/'status.json').read_text()))
records=[json.loads(line) for line in (run/'results.jsonl').read_text().splitlines() if line.strip()] if (run/'results.jsonl').exists() else []
trials=pd.DataFrame([{**r['config'],'selection_score':r['selection_score']} for r in records])
if len(trials):
    display(trials.groupby(['objective','horizon']).selection_score.agg(['count','max','median']))

In [ ]:
if len(trials):
    fig=px.scatter(trials,x='id',y='selection_score',color='history',facet_row='objective',facet_col='horizon',height=1200,title='Development score: сравнение внутри задачи, выше лучше')
    fig.update_yaxes(matches=None)
    fig.show()

In [ ]:
if (run/'final_metrics.csv').exists():
    final=pd.read_csv(run/'final_metrics.csv')
    display(final)
else:
    print('Финальная оценка пока не завершена. Промежуточный score не является качеством на evaluation.')

In [ ]:
if (run/'final_metrics.csv').exists():
    risk=final[(final.objective=='risk') & final.tp.notna()].copy()
    if len(risk):
        risk['recall']=risk.tp/(risk.tp+risk.fn)
        risk['precision']=risk.tp/(risk.tp+risk.fp)
        risk['false_positive_rate']=risk.fp/(risk.fp+risk.tn)
        display(risk[['year','horizon','evaluation_ap','evaluation_prevalence','recall','precision','false_positive_rate','brier','baseline_brier']])
        px.bar(risk,x='horizon',y=['recall','precision','false_positive_rate'],facet_col='year',barmode='group',title='Классификатор превышения: фактические ошибки').show()

In [ ]:
if (run/'health.jsonl').exists():
    health=pd.read_json(run/'health.jsonl',lines=True)
    parts=health.gpu_util_mem_temp_power.str.split(',',expand=True)
    if parts.shape[1]==4:
        health['gpu_util_percent']=pd.to_numeric(parts[0],errors='coerce')
        px.line(health,x='utc',y='gpu_util_percent',title='Загрузка A100 во времени').show()

## Ограничения
Задержки ЛИМС условны. Исторические периоды ранее изучались. Длительный подбор повышает риск переобучения на validation: проверяйте устойчивость и не выбирайте конфигурацию по evaluation. Модель риска не задаёт управляющие воздействия. Признаки с неопределённым физическим смыслом требуют проверки перед внедрением.